# 06 — Classical Unsupervised ML

## Objective
Benchmark Isolation Forest and LOF on the controlled feature matrix. Training is label-free; validation labels are used only for model comparison.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Train detectors on train

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); train,val,test,_=chronological_split(fraud); b=HistoryFeatureBuilder().fit(train); Xtr,Xv,Xt=map(b.transform,[train,val,test]); cols=feature_columns(Xtr); pre=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]); A=pre.fit_transform(Xtr[cols]); B=pre.transform(Xv[cols]); models=fit_models(A,lof_sample=30000); Xv["iforest_score"],Xv["lof_score"]=model_scores(models,B)

## 2. Validation ranking benchmark

In [2]:
rows=[]
for name,c in [("Isolation Forest","iforest_score"),("LOF","lof_score")]:
    m=ranking_metrics(Xv['class'],Xv[c]); m["model"]=name; rows.append(m)
display(pd.DataFrame(rows)); px.bar(pd.DataFrame(rows),x="model",y="pr_auc",title="Validation PR-AUC").show()
import joblib; joblib.dump(pre,ART/"classical_preprocessor.joblib"); joblib.dump(models,ART/"classical_models.joblib")

,pr_auc,roc_auc,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_500,recall_at_500,model
0,0.045658,0.499626,0.00,0.000000,0.04,0.003846,0.046,0.022115,Isolation Forest
1,0.043466,0.486570,0.04,0.001923,0.03,0.002885,0.028,0.013462,LOF


['D:\\fraud_ecommerce_graph_anomaly_and_ml\\artifacts\\classical_models.joblib']

## 3. Interactive investigation queue

In [3]:
score=widgets.Dropdown(options=["iforest_score","lof_score"],
                       value="iforest_score",
                       description="Score"); 
pct=widgets.IntSlider(value=95,min=80,max=99,description="Top %"); out=widgets.Output()
def inspect(*_):
    q=np.percentile(Xv[score.value],pct.value); view=Xv[Xv[score.value]>=q].sort_values(score.value,ascending=False)
    with out: out.clear_output(); display(view[["user_id","device_id","ip_address","purchase_value",score.value,"class"]].head(30))
score.observe(inspect,'value'); pct.observe(inspect,'value'); display(widgets.HBox([score,pct]),out); inspect()

Output()